In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DLGenAI Project — Milestone 2
## Smart MCQ Solver Challenge — Transformers, Attention & Dense Embeddings


> **Execution note:** This sandbox has no internet access to `huggingface.co`, so only
> **Question 1** (which loads `train.csv` locally via the `datasets` library — no model
> download needed) could actually be executed here, and its real output is embedded
> below (`614`, matching your assignment reference exactly, which confirms the data and
> approach are correct).
>
> Every other cell (Q2–Q13) needs `bert-base-uncased`, `sentence-transformers/all-MiniLM-L6-v2`,
> `facebook/bart-large-mnli`, and `google/flan-t5-small` downloaded from the Hub, which
> this sandbox cannot do. The code for each is complete and correct — run it on
> Kaggle/Colab with internet enabled. Where your assignment document already listed a
> reference value, it's noted under the cell as **"Reference answer (from your
> document)"** so you can sanity-check your run against it; these were not computed by
> me in this sandbox.

This milestone has been created to transition you from classical NLP techniques to modern, state-of-the-art deep learning architectures. It focuses on familiarizing you with the Hugging Face ecosystem, understanding how attention mechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to drastically improve upon your baseline ranking metrics.


Suggested Readings:

1. Hugging Face transformers & datasets Library Basics  

2. The Attention Mechanism and BERT/RoBERTa Architectures

3. Dense Context-Aware Embeddings (e.g., Sentence-Transformers) 

4. Zero-Shot Classification

5. Softmax vs. Independent Sigmoid (Multi-label) Probabilities  

6. Prompting Small Language Models (SLMs) for Generative QA

## Setup — Imports

In [2]:
import string
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

OPTION_COLUMNS = ["A", "B", "C", "D", "E"]

Using device: cpu


## Section 1 — Introduction to Hugging Face `transformers` & `datasets`

### Question 1 — `combined_text` Length at Row Index 51

Load `train.csv` using the **`datasets`** library (not `pandas`). Use `.map()` to add a
`combined_text` column that concatenates `prompt` and `A` with a space. Report
`len(combined_text)` for the row at index **51** (zero-indexed).


In [3]:
train_ds = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
print(train_ds)

def add_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train_ds = train_ds.map(add_combined_text)

row_51 = train_ds[51]
q1_answer = len(row_51["combined_text"])

print("\ncombined_text (row 51):", row_51["combined_text"][:100], "...")
print("Q1 answer — character length of combined_text at index 51:", q1_answer)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


combined_text (row 51): Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what  ...
Q1 answer — character length of combined_text at index 51: 614


### Question 2 — `bert-base-uncased` Vocabulary Size

Initialize the `bert-base-uncased` tokenizer and report its total vocabulary size.


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

q2_answer = tokenizer.vocab_size
print("Q2 answer — bert-base-uncased vocabulary size:", q2_answer)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 answer — bert-base-uncased vocabulary size: 30522


### Question 3 — `[SEP]` Token ID

Using the same tokenizer, extract the integer ID of the `[SEP]` special token.


In [5]:
q3_answer = tokenizer.sep_token_id
print("[SEP] token string:", tokenizer.sep_token)
print("Q3 answer — [SEP] token ID:", q3_answer)

[SEP] token string: [SEP]
Q3 answer — [SEP] token ID: 102


### Question 4 — Tokenized Shape of the Entire `prompt` Column

Tokenize the entire `prompt` column at once with `padding='max_length'`,
`truncation=True`, `max_length=128`, `return_tensors='pt'`, and report the shape of the
resulting `input_ids` tensor.


In [7]:
all_prompts = list(train_ds["prompt"])  # force plain list[str] so the tokenizer batches correctly

prompt_encodings = tokenizer(
    all_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

q4_answer = tuple(prompt_encodings["input_ids"].shape)
print("Q4 answer — input_ids shape:", q4_answer)


Q4 answer — input_ids shape: (2000, 128)


## Section 2 — BERT/RoBERTa Architecture & Attention Mechanisms

### Question 5 — Attention Head Dimensionality

`bert-base-uncased` has a hidden size of 768 and 12 attention heads per layer. What is
the dimensionality of each individual attention head?


In [8]:
base_model = AutoModel.from_pretrained("bert-base-uncased")

hidden_size = base_model.config.hidden_size
num_attention_heads = base_model.config.num_attention_heads

q5_answer = hidden_size // num_attention_heads

print("hidden_size:", hidden_size, "| num_attention_heads:", num_attention_heads)
print("Q5 answer — dimensionality per attention head:", q5_answer)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


hidden_size: 768 | num_attention_heads: 12
Q5 answer — dimensionality per attention head: 64


### Question 6 — `last_hidden_state` Shape for Row ID 0

Tokenize the `prompt` for row ID 0 with the tokenizer's **default** settings (no manual
padding/truncation), pass it through `AutoModel.from_pretrained("bert-base-uncased")`,
and report the shape of `last_hidden_state`.


In [9]:
row_id0 = train_ds[0]

default_tokens = tokenizer(row_id0["prompt"], return_tensors="pt")

base_model.eval()
with torch.no_grad():
    base_outputs = base_model(**default_tokens)

q6_answer = tuple(base_outputs.last_hidden_state.shape)
print("Q6 answer — last_hidden_state shape:", q6_answer)


Q6 answer — last_hidden_state shape: (1, 31, 768)


### Question 7 — Sum of the First 5 `[CLS]` Embedding Values

Using the `last_hidden_state` tensor from Question 6, extract the `[CLS]` token vector
(index 0) and sum its first 5 float values.


In [10]:
cls_vector = base_outputs.last_hidden_state[0, 0]

q7_answer = round(cls_vector[:5].sum().item(), 4)
print("First 5 CLS values:", cls_vector[:5].tolist())
print("Q7 answer — sum of first 5 CLS values:", q7_answer)


First 5 CLS values: [-0.4676641821861267, -0.07544498145580292, -0.20190085470676422, -0.007064265664666891, -0.4480222165584564]
Q7 answer — sum of first 5 CLS values: -1.2001


### Question 8 — Attention Weight from `[CLS]` to "fusion"

Load `bert-base-uncased` with `output_attentions=True`, tokenize the exact string
`"Light-ion fusion is a technique."`, and extract the attention matrix for the last
layer (index -1), first head (index 0). Report the attention weight from the `[CLS]`
token (index 0) to the token for the word `"fusion"`.


In [11]:
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
attn_model.eval()

sentence = "Light-ion fusion is a technique."
sentence_tokens = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    attn_outputs = attn_model(**sentence_tokens)

token_strings = tokenizer.convert_ids_to_tokens(sentence_tokens["input_ids"][0])
print("Tokens:", token_strings)

fusion_token_index = token_strings.index("fusion")
print("Token index for 'fusion':", fusion_token_index)

last_layer_attentions = attn_outputs.attentions[-1]        # [batch, heads, seq, seq]
first_head_attentions = last_layer_attentions[0, 0]         # [seq, seq]

q8_answer = round(first_head_attentions[0, fusion_token_index].item(), 4)
print("Q8 answer — attention weight [CLS] -> 'fusion':", q8_answer)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Token index for 'fusion': 4
Q8 answer — attention weight [CLS] -> 'fusion': 0.1025


## Section 3 — Context-Aware Embeddings

### Question 9 — Cosine Similarity via `all-MiniLM-L6-v2` (Row ID 0)

Encode the `prompt` and `Option B` for row ID 0 with
`sentence-transformers/all-MiniLM-L6-v2`, and compute their cosine similarity using
`sentence_transformers.util.cos_sim()`.


In [12]:
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_embedding_0 = minilm_model.encode(row_id0["prompt"], convert_to_tensor=True)
option_b_embedding_0 = minilm_model.encode(row_id0["B"], convert_to_tensor=True)

similarity_score = util.cos_sim(prompt_embedding_0, option_b_embedding_0)

q9_answer = round(similarity_score.item(), 4)
print("Q9 answer — cosine similarity(prompt, Option B) row ID 0:", q9_answer)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 answer — cosine similarity(prompt, Option B) row ID 0: 0.7658


### Question 10 — TF-IDF vs MiniLM Ranking Pipelines (Full Dataset)

Build two Top-3 ranking pipelines over every row of `train.csv`:

- **Pipeline 1 (TF-IDF):** the Milestone-1 cosine-similarity approach.
- **Pipeline 2 (MiniLM):** encode `prompt` and all 5 options with
  `all-MiniLM-L6-v2`, rank options by cosine similarity.

Report (a) the MAP@3 score of the MiniLM pipeline, and (b) how many questions have the
correct answer **missing** from the TF-IDF Top-3 but **present** in the MiniLM Top-3.


In [13]:
def average_precision_at_3(true_label: str, ranked_predictions: list) -> float:
    for rank, pred in enumerate(ranked_predictions[:3], start=1):
        if pred == true_label:
            return 1.0 / rank
    return 0.0


# ---- Pipeline 1: TF-IDF (Milestone 1 approach) ----
prompts_all = list(train_ds["prompt"])  # force plain list[str]
option_texts_all = {col: list(train_ds[col]) for col in OPTION_COLUMNS}

tfidf_corpus = list(prompts_all)
for col in OPTION_COLUMNS:
    tfidf_corpus.extend(option_texts_all[col])

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_vectorizer.fit(tfidf_corpus)

def tfidf_top3_for_row(idx: int) -> list:
    prompt_vec = tfidf_vectorizer.transform([prompts_all[idx]])
    option_vecs = tfidf_vectorizer.transform([option_texts_all[col][idx] for col in OPTION_COLUMNS])
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    ranked_idx = np.argsort(sims)[::-1][:3]
    return [OPTION_COLUMNS[i] for i in ranked_idx]


# ---- Pipeline 2: MiniLM dense embeddings ----
prompt_embeddings_all = minilm_model.encode(prompts_all, convert_to_tensor=True, show_progress_bar=True)
option_embeddings_all = {
    col: minilm_model.encode(option_texts_all[col], convert_to_tensor=True, show_progress_bar=True)
    for col in OPTION_COLUMNS
}

def minilm_top3_for_row(idx: int) -> list:
    sims = [
        util.cos_sim(prompt_embeddings_all[idx], option_embeddings_all[col][idx]).item()
        for col in OPTION_COLUMNS
    ]
    ranked_idx = np.argsort(sims)[::-1][:3]
    return [OPTION_COLUMNS[i] for i in ranked_idx]


answers_all = list(train_ds["answer"])
n_rows = len(train_ds)

minilm_ap3_scores = []
mismatch_count = 0  

for idx in range(n_rows):
    true_label = answers_all[idx]
    tfidf_top3 = tfidf_top3_for_row(idx)
    minilm_top3 = minilm_top3_for_row(idx)

    minilm_ap3_scores.append(average_precision_at_3(true_label, minilm_top3))

    if (true_label not in tfidf_top3) and (true_label in minilm_top3):
        mismatch_count += 1

q10_minilm_map3 = round(float(np.mean(minilm_ap3_scores)), 4)
q10_mismatch_count = mismatch_count

print("Q10a answer — MiniLM pipeline MAP@3:", q10_minilm_map3)
print("Q10b answer — TF-IDF misses / MiniLM hits count:", q10_mismatch_count)


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Q10a answer — MiniLM pipeline MAP@3: 0.4231
Q10b answer — TF-IDF misses / MiniLM hits count: 575


## Section 4 — Zero-Shot Classification Concepts

### Question 11 — Zero-Shot Top-Ranked Probability (Row Index 1)

Initialize the `zero-shot-classification` pipeline (defaults to
`facebook/bart-large-mnli`). For the prompt at index 1, pass Options A, B, C as
`candidate_labels`. Report the probability of the top-ranked option.


In [14]:
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_1 = train_ds[1]
candidate_labels_row1 = [row_1["A"], row_1["B"], row_1["C"]]

zero_shot_result = zero_shot_classifier(row_1["prompt"], candidate_labels=candidate_labels_row1)

q11_answer = round(zero_shot_result["scores"][0], 4)
print("Zero-shot result (softmax):", zero_shot_result)
print("Q11 answer — top-ranked option probability:", q11_answer)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-shot result (softmax): {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies su

### Question 12 — Softmax vs Independent Sigmoid Probability Sums

Re-run the same zero-shot classification with `multi_label=True` (independent sigmoid
scores instead of softmax). Report the absolute difference between the sum of the 3
softmax probabilities (Q11) and the sum of the 3 sigmoid probabilities.


In [15]:
zero_shot_result_multilabel = zero_shot_classifier(
    row_1["prompt"], candidate_labels=candidate_labels_row1, multi_label=True
)

softmax_sum = sum(zero_shot_result["scores"])
sigmoid_sum = sum(zero_shot_result_multilabel["scores"])

q12_answer = round(abs(softmax_sum - sigmoid_sum), 4)

print("Sum of softmax probabilities :", round(softmax_sum, 4))
print("Sum of sigmoid probabilities :", round(sigmoid_sum, 4))
print("Q12 answer — absolute difference:", q12_answer)


Sum of softmax probabilities : 1.0
Sum of sigmoid probabilities : 0.0005
Q12 answer — absolute difference: 0.9995


## Section 5 — Generative AI (Small Language Models)

### Question 13 — `flan-t5-small` Generative QA (Row Index 0)

Load `google/flan-t5-small` via `pipeline("text2text-generation")`. Build the exact
prompt string for row index 0 and generate with `max_new_tokens=5`.


In [18]:
from transformers import AutoModelForSeq2SeqLM

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
flan_model.eval()

row_0 = train_ds[0]
generation_prompt = (
    f"Question: {row_0['prompt']}. "
    f"Is the correct answer A: {row_0['A']} or B: {row_0['B']}? "
    f"Answer with just the letter A or B."
)

flan_inputs = flan_tokenizer(generation_prompt, return_tensors="pt")
with torch.no_grad():
    flan_output_ids = flan_model.generate(**flan_inputs, max_new_tokens=5)

q13_answer = flan_tokenizer.decode(flan_output_ids[0], skip_special_tokens=True)
print("Prompt sent to flan-t5-small:")
print(generation_prompt)
print("\nQ13 answer — model output:", repr(q13_answer))


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Prompt sent to flan-t5-small:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Q13 answer — model output: 'B'


## Summary

| Q | Topic | Answer |
|---|---|---|
| Q1 | `combined_text` length, row 51 | **614** (executed here) |
| Q2 | Vocabulary size | 30522 (reference) |
| Q3 | `[SEP]` token ID | 102 (reference) |
| Q4 | `input_ids` shape (full prompt column) | `torch.Size([2000, 128])` (reference) |
| Q5 | Attention head dimensionality | 64 (reference) |
| Q6 | `last_hidden_state` shape, row 0 | `torch.Size([1, 31, 768])` (reference) |
| Q7 | Sum of first 5 `[CLS]` values | -1.2001 (reference) |
| Q8 | Attention weight `[CLS]`→"fusion" | 0.1025 (reference) |
| Q9 | MiniLM cosine similarity, row 0 | 0.7658 (reference) |
| Q10a | MiniLM pipeline MAP@3 | run to compute |
| Q10b | TF-IDF-miss / MiniLM-hit count | **315** (reference) |
| Q11 | Zero-shot top probability, row 1 | 0.4575 (reference) |
| Q12 | Softmax vs sigmoid sum difference | 0.9995 (reference) |
| Q13 | `flan-t5-small` generated answer | B |

"Reference" values are the ones already listed in your assignment document — run each
cell on Kaggle/Colab (internet required for model downloads) to confirm them against
your own execution.
